In [17]:
from bs4 import SoupStrainer, BeautifulSoup
import pandas as pd
import io
import re

In [ ]:
# Transform HTML into single DataFrame

try:
    with open('../html/output.html', 'r') as f:
        html = f.read()
    soup = BeautifulSoup(html, features='lxml')

except FileNotFoundError as e:
    print('output.html not found, run fetch_finals_schedule tool')

tables = soup.find_all('table')
dfs = []
for table in tables:
    df = pd.read_html(io.StringIO(str(table)))[0]

    caption = table.find('caption')
    caption = caption.text.strip() if caption else None
    df['Finals Day'] = caption

    dfs.append(df)

df = pd.concat(dfs)
df.head()


,Class Time,Class Day(s),Final Meeting Time,Finals Day
0,8:30 a.m.,Tuesday/Thursday (TR),8-10 a.m.,"Thursday, First day of finals"
1,9 a.m.,Tuesday/Thursday (TR) or Thursday (R),8-10 a.m.,"Thursday, First day of finals"
2,11:30 a.m.,Tuesday/Thursday (TR),10:15 a.m.-12:15 p.m.,"Thursday, First day of finals"
3,12 p.m.,Tuesday/Thursday (TR),10:15 a.m.-12:15 p.m.,"Thursday, First day of finals"
4,1:30 p.m.,Thursday (R),12:45-2:45 p.m.,"Thursday, First day of finals"


In [19]:
# Standardize Class Day(s)

def extract_codes(str):
    return re.findall(r'\(([A-Z]+)\)', str)

df['Class Day(s)'] = df['Class Day(s)'].apply(extract_codes)
df = df.explode('Class Day(s)').reset_index(drop=True)
df.head()


,Class Time,Class Day(s),Final Meeting Time,Finals Day
0,8:30 a.m.,TR,8-10 a.m.,"Thursday, First day of finals"
1,9 a.m.,TR,8-10 a.m.,"Thursday, First day of finals"
2,9 a.m.,R,8-10 a.m.,"Thursday, First day of finals"
3,11:30 a.m.,TR,10:15 a.m.-12:15 p.m.,"Thursday, First day of finals"
4,12 p.m.,TR,10:15 a.m.-12:15 p.m.,"Thursday, First day of finals"


In [20]:
# Export generic_finals_schedule.csv

# df.to_csv('../data/generic_finals_schedule.csv')

In [21]:
# Import input.csv

try:
    student_df = pd.read_csv('../data/input.csv')
    
except FileNotFoundError as e:
    print(e)
    print('Using sample data instead')
    student_df = pd.read_csv('../data/sample_input.csv')

student_df.head()

[Errno 2] No such file or directory: '../data/input.csv'
Using sample data instead


,Class Name,Class Time,Class Day(s)
0,CS422,9 a.m.,TR
1,CS425,10:30 a.m.,TR
2,CS454,5:30 p.m.,MW
3,CS457,2:30 p.m.,MW
4,ENGR301,4:30 p.m.,TR


In [22]:
# Calculate finals schedule (output)

final_schedule_df = student_df.merge(right=df)
final_schedule_df.drop(['Class Time','Class Day(s)'], axis='columns', inplace=True)
final_schedule_df

,Class Name,Final Meeting Time,Finals Day
0,CS422,8-10 a.m.,"Thursday, First day of finals"
1,CS425,10:15 a.m.-12:15 p.m.,"Tuesday, Fourth Day of Finals"
2,CS454,5:30-7:30 p.m.,"Monday, Third Day of Finals"
3,CS457,3-5 p.m.,"Monday, Third Day of Finals"
4,ENGR301,3-5 p.m.,"Tuesday, Fourth Day of Finals"


In [23]:
# Export output.csv

final_schedule_df.to_csv('../data/output.csv')